# 07 — 动态断裂韧性指标

**BWV861 音乐形式化分析系统** · Step 7

计算全曲所有主题终止事件的动态指标:
- A_i(t) 记忆激活
- B_dyn 动态断裂势能
- E_cont 延续预期
- U_cad 句法未闭合度
- C_res 残余相干
- D_dyn 动态净断裂感
- T_m 音乐结构韧性

In [4]:
import sys, os, pickle
import numpy as np
PROJECT_ROOT = r"e:\大三\Cline Projects\bien_project"
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

# 加载过滤后的主题数据
with open("data/themes_filtered.pkl", "rb") as f:
    filtered = pickle.load(f)
occurrences = filtered["occurrences"]
strict_types = filtered["strict_types"]
symmetric_families = filtered["symmetric_families"]
gains = filtered.get("gains", {})
importance = filtered.get("importance", {})

# 加载 06 重新计算的相似度和网络 (将使用过滤后数据重新运行)
from music_analysis.distance import load_similarity
from music_analysis.network import load_network
from music_analysis.dynamics import (
    compute_temporal_constants, compute_all_dynamic_metrics,
    save_dynamic_metrics,
)

dist_matrix, sim_matrix, type_ids, ell = load_similarity()
G, breakage = load_network()

print(f"✅ 已加载过滤后数据: {len(occurrences)} occurrences, "
      f"{len(strict_types)} strict types, {G.number_of_nodes()} nodes")

✅ 已加载过滤后数据: 268 occurrences, 52 strict types, 52 nodes


In [5]:
# 时间常量
temporal = compute_temporal_constants(occurrences)

# 全部动态指标
metrics = compute_all_dynamic_metrics(
    occurrences, strict_types, G, sim_matrix, type_ids, temporal
)

save_dynamic_metrics(metrics)

# Top-5 最强断裂事件
top_events = sorted(metrics, key=lambda m: -m["D_dyn"])[:5]
print(f"\nTop-5 D_dyn 断裂事件:")
for m in top_events:
    print(f"  事件#{m['occ_id']} (T_{m['strict_type_id']}): "
          f"D_dyn={m['D_dyn']:.4f}, B_dyn={m['B_dyn']:.4f}, "
          f"E_cont={m['E_cont']:.3f}, U_cad={m['U_cad']:.3f}, "
          f"C̃_res={m['C_tilde']:.3f}")

τ_𝔇 = 2.50 QL  (median dur=1.50 + median gap=1.00)
正在计算 268 次终止事件的动态指标...
✅ 268 次终止事件的动态指标已计算
  D_dyn: min=0.0000, max=0.0933, mean=0.0137, median=0.0000
✅ 动态指标已保存: data/dynamic_metrics.pkl

Top-5 D_dyn 断裂事件:
  事件#380 (T_14): D_dyn=0.0933, B_dyn=0.3624, E_cont=1.000, U_cad=0.333, C̃_res=0.228
  事件#389 (T_8): D_dyn=0.0931, B_dyn=0.6062, E_cont=1.000, U_cad=0.167, C̃_res=0.078
  事件#461 (T_4): D_dyn=0.0855, B_dyn=0.8511, E_cont=1.000, U_cad=0.167, C̃_res=0.397
  事件#379 (T_13): D_dyn=0.0767, B_dyn=0.2700, E_cont=1.000, U_cad=0.333, C̃_res=0.147
  事件#874 (T_27): D_dyn=0.0742, B_dyn=0.4139, E_cont=0.639, U_cad=0.333, C̃_res=0.159


## 7.1 计算逻辑调试 — 逐项追踪 D_dyn 的各因子

选取一个有非零 B_dyn 的事件，手动追踪 $D_{dyn} = B_{dyn} \cdot E_{cont} \cdot U_{cad} \cdot (1-\tilde{C}_{res})$

In [3]:
tau = temporal["tau"]
typical_durs = temporal["typical_durs"]

# 选取有代表性的事件进行追踪
top_bdyn = sorted(metrics, key=lambda m: -m["B_dyn"])[:5]

print("=" * 60)
print("🔍 D_dyn 计算逻辑逐要素审查")
print("公式: D_dyn = B_dyn × E_cont × U_cad × (1 - C̃_res)")
print("=" * 60)

for rank, m in enumerate(top_bdyn, 1):
    tid = m["strict_type_id"]
    t_c = m["t_c"]

    # 查 T_i 的所有出现时长
    occs_of_type = strict_types.get(tid, [])
    durs = [o["sigma"][1] - o["sigma"][0] for o in occs_of_type]
    D_typ = typical_durs.get(tid, -1)
    unique_durs = sorted(set(round(d, 3) for d in durs))

    # 找对应的 occurrence
    occ = next((o for o in occurrences if o["occ_id"] == m["occ_id"]), None)
    dur_this = occ["sigma"][1] - occ["sigma"][0] if occ else -1

    print(f"\n--- 事件#{m['occ_id']} T_{tid} (出{len(occs_of_type)}次) t_c={t_c:.1f} ---")
    print(f"  T_{tid} 的所有时长: {unique_durs} (共{len(durs)}次)")
    print(f"  D_typ (median) = {D_typ:.3f}")
    print(f"  本次 dur = {dur_this:.3f}")
    print(f"  → dur/D_typ = {dur_this/D_typ:.3f} → U_cad = {m['U_cad']:.4f}")
    print(f"  B_dyn = {m['B_dyn']:.4f}")
    print(f"  E_cont = {m['E_cont']:.4f}")
    print(f"  C̃_res = {m['C_tilde']:.4f}  → (1-C̃_res) = {1-m['C_tilde']:.4f}")
    product = m["B_dyn"] * m["E_cont"] * m["U_cad"] * (1 - m["C_tilde"])
    print(f"  D_dyn = {m['B_dyn']:.4f}×{m['E_cont']:.4f}×{m['U_cad']:.4f}×{1-m['C_tilde']:.4f} = {product:.6f}")
    if m["U_cad"] == 0:
        print(f"  ⚠️  U_cad=0 导致 D_dyn=0")

# 统计 U_cad 分布
u_values = [m["U_cad"] for m in metrics]
print(f"\n{'='*60}")
print(f"📊 U_cad 分布统计 (共{len(u_values)}次终止事件):")
print(f"  U_cad=0 的事件数: {sum(1 for u in u_values if u < 1e-9)} / {len(u_values)}")
print(f"  U_cad>0 的事件数: {sum(1 for u in u_values if u > 1e-9)} / {len(u_values)}")
print(f"  max(U_cad) = {max(u_values):.4f}")

# 也查查 B_dyn 的分布
b_values = [m["B_dyn"] for m in metrics]
print(f"\n📊 B_dyn 分布统计:")
print(f"  B_dyn=0 的事件数: {sum(1 for b in b_values if b < 1e-9)} / {len(b_values)}")
print(f"  B_dyn>0 的事件数: {sum(1 for b in b_values if b > 1e-9)} / {len(b_values)}")
print(f"  max(B_dyn) = {max(b_values):.4f}")

# 最终结论
print(f"\n{'='*60}")
print(f"📋 诊断结论:")
print(f"  D_dyn ≈ 0 的根本原因: U_cad 全为 0")
print(f"  原因: 过滤后所有主题类型的每次出现时长 = 该类型的典型时长")
print(f"  → 所有出现都是'完整陈述'，不触发句法未闭合度")
print(f"  → D_dyn = B_dyn × E_cont × 0 × (1-C̃_res) = 0")
print(f"{'='*60}")

🔍 D_dyn 计算逻辑逐要素审查
公式: D_dyn = B_dyn × E_cont × U_cad × (1 - C̃_res)

--- 事件#781 T_2 (出11次) t_c=163.8 ---
  T_2 的所有时长: [1.25] (共11次)
  D_typ (median) = 1.250
  本次 dur = 1.250
  → dur/D_typ = 1.000 → U_cad = 0.1667
  B_dyn = 1.1025
  E_cont = 0.4684
  C̃_res = 0.2356  → (1-C̃_res) = 0.7644
  D_dyn = 1.1025×0.4684×0.1667×0.7644 = 0.065786

--- 事件#1010 T_2 (出11次) t_c=203.8 ---
  T_2 的所有时长: [1.25] (共11次)
  D_typ (median) = 1.250
  本次 dur = 1.250
  → dur/D_typ = 1.000 → U_cad = 0.1667
  B_dyn = 0.9966
  E_cont = 0.5034
  C̃_res = 0.7249  → (1-C̃_res) = 0.2751
  D_dyn = 0.9966×0.5034×0.1667×0.2751 = 0.023002

--- 事件#468 T_2 (出11次) t_c=103.8 ---
  T_2 的所有时长: [1.25] (共11次)
  D_typ (median) = 1.250
  本次 dur = 1.250
  → dur/D_typ = 1.000 → U_cad = 0.1667
  B_dyn = 0.9795
  E_cont = 0.5787
  C̃_res = 0.6523  → (1-C̃_res) = 0.3477
  D_dyn = 0.9795×0.5787×0.1667×0.3477 = 0.032852

--- 事件#746 T_3 (出11次) t_c=158.0 ---
  T_3 的所有时长: [1.25] (共11次)
  D_typ (median) = 1.250
  本次 dur = 1.250
  → dur/D_typ =